In [ ]:
!pip install torch_geometric

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors
from torch_geometric.data import Data, Dataset
import numpy as np
from torch_geometric.loader import DataLoader
import torch.optim as optim
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, GraphNorm
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.nn import radius_graph

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import pandas as pd
# Set pandas to show all columns for debugging
pd.set_option('display.max_columns', None)

# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/202509_CURRENT_1024x1024dataset.csv', index_col=0)

# Parse columns
unique_region = df["unique_region"].values
labels = df["Cell Type"].values

# Cell positions
x_positions = df['x'].to_numpy()
y_positions = df['y'].to_numpy()
print(x_positions)

# Drop everything not used in GCN features, Change this based on your spatial omics data
drop_cols = [
    "Cell Type", "unique_region", "donor", "array", "Xcorr", "Ycorr",
    "Tissue_location", "tissue", "region", "OLFM4", "FAP", "CD25",
    "CK7", "MUC6", "Cell Type em", "Cell subtype", "Neighborhood",
    "Neigh_sub", "Neighborhood_Ind", "NeighInd_sub", "Community",
    "Major Community", "Tissue Segment", "Tissue Unit", "CollIV"
]
features_df = df.drop(columns=drop_cols)
feature_cols = features_df.columns

In [ ]:
def build_radius_edges(positions, radius):
    """
    Build edge_index using only NumPy — does NOT require torch-cluster.
    positions: (N,2) array
    radius: float
    """
    N = positions.shape[0]
    edge_list = []

    # Compute pairwise distances (O(N^2))
    dist_mat = np.linalg.norm(
        positions[:, None, :] - positions[None, :, :],
        axis=-1
    )

    # Add edges for pairs within radius
    for i in range(N):
        for j in range(N):
            if i != j and dist_mat[i, j] <= radius:
                edge_list.append([i, j])

    if len(edge_list) == 0:
        return torch.empty((2, 0), dtype=torch.long)

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    return edge_index


In [ ]:
class RegionGraphDataset(Dataset):
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, idx):
        return self.data_list[idx]
    def __init__(
        self,
        df,
        feature_cols,
        label_col='Cell Type',
        region_col='unique_region',
        pos_cols=['x', 'y'],
        k_neighbors=20
    ):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.region_col = region_col
        self.pos_cols = pos_cols
        self.k = k_neighbors

        # Label encoding
        self.label_encoder = LabelEncoder()
        self.df[label_col] = self.label_encoder.fit_transform(self.df[label_col])
        self.label_col = label_col

        self.region_ids = self.df[self.region_col].unique()
        self.data_list = []

        for region_id in self.region_ids:
            region_df = self.df[self.df[self.region_col] == region_id]

            features = region_df[self.feature_cols].values
            labels = region_df[self.label_col].values
            positions = region_df[self.pos_cols].values

            num_nodes = features.shape[0]

            if num_nodes <= 1:
                edge_index = torch.empty((2, 0), dtype=torch.long)

            else:
                # ---------- KNN graph ----------
                knn = NearestNeighbors(
                    n_neighbors=min(self.k + 1, num_nodes),
                    algorithm='ball_tree'
                )
                knn.fit(positions)

                distances, indices = knn.kneighbors(positions)

                # Build directed edges
                edge_list = []
                for i, nbrs in enumerate(indices):
                    for nbr in nbrs:
                        if nbr != i:
                            edge_list.append([i, nbr])

                edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

                # ---------- Make graph undirected (symmetric KNN) ----------
                edge_index = torch.cat(
                    [edge_index, edge_index.flip(0)],
                    dim=1
                ).unique(dim=1)

            # Create Data object
            x = torch.tensor(features, dtype=torch.float32)
            y = torch.tensor(labels, dtype=torch.long)

            data = Data(x=x, edge_index=edge_index, y=y)
            data.region_id = region_id

            self.data_list.append(data)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import APPNP, GraphNorm

class GCNClassifier(nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels,
        num_classes,
        dropout=0.1,
        alpha=0.9,      # teleport probability → higher = stronger self-feature weight
        K=20            # number of propagation steps
    ):
        super().__init__()

        # MLP before propagation (APPNP standard)
        self.lin1 = nn.Linear(in_channels, hidden_channels)
        self.norm1 = GraphNorm(hidden_channels)

        self.lin2 = nn.Linear(hidden_channels, hidden_channels)
        self.norm2 = GraphNorm(hidden_channels)

        # APPNP propagation module
        self.propagation = APPNP(
            K=K,
            alpha=alpha,
            dropout=dropout
        )

        self.dropout = dropout
        self.out_lin = nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index):
        # ----- MLP encoder -----
        x = self.lin1(x)
        x = self.norm1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.lin2(x)
        x = self.norm2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # ----- APPNP propagation (self-feature preserving) -----
        x = self.propagation(x, edge_index)

        # ----- Classification head -----
        x = self.out_lin(x)
        return x


In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index)
            pred = out.argmax(dim=1)
            correct += (pred == data.y).sum().item()
            total += data.y.size(0)
    return correct / total


In [ ]:
# Entrance Training Pipeline

# Define input feature columns
feature_cols = feature_cols   # replace with your actual feature list

# Create dataset using radius-graph version
dataset = RegionGraphDataset(
    df=df,
    feature_cols=feature_cols,
)

# Graph-level batching: each batch = one region graph
train_loader = DataLoader(dataset, batch_size=1, shuffle=True)

# Select device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize GNN model
model = GCNClassifier(
    in_channels=len(feature_cols),
    hidden_channels=768,
    num_classes=len(np.unique(df['Cell Type']))
).to(device)

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

# Training loop
for epoch in range(1, 40):
    # Train for one epoch
    loss = train_one_epoch(model, train_loader, optimizer, device)
    # Evaluate on training set (node-level accuracy)
    acc = evaluate(model, train_loader, device)
    print(f"Epoch {epoch}, Loss {loss:.4f}, Train Acc {acc:.4f}")


In [ ]:
# Inference on graph dataset

dataset = RegionGraphDataset(
    df=df,
    feature_cols=feature_cols,
    k_neighbors=20
)

loader = DataLoader(dataset, batch_size=1, shuffle=False)

model.eval()
all_probs = []
all_x = []
all_y = []
all_region = []

with torch.no_grad():
    for batch in loader:
        # Each batch is ONE region graph
        batch = batch.to(device)

        # GNN forward pass
        out = model(batch.x, batch.edge_index)  # [num_nodes, num_classes]

        # softmax probability for each node
        probs = torch.softmax(out, dim=1).cpu().numpy()

        # store
        all_probs.append(probs)
        all_x.append(batch.x.cpu().numpy())    # original features not needed usually
        all_y.append(batch.y.cpu().numpy())
        all_region.extend([batch.region_id] * probs.shape[0])


# ----------------------------------------
# Concatenate results across all regions
# ----------------------------------------

all_probs = np.concatenate(all_probs, axis=0)   # shape: [num_cells_total, num_classes]

# Build dataframe
result_df = pd.DataFrame(
    all_probs,
    columns=[f"prob_class{i}" for i in range(all_probs.shape[1])]
)

# Add coordinates and region info
result_df["x"] = df["x"].values
result_df["y"] = df["y"].values
result_df["region"] = df["unique_region"].values

# Order columns nicely
cols = ["x", "y", "region"] + [f"prob_class{i}" for i in range(all_probs.shape[1])]
result_df = result_df[cols]

# Save
result_df.to_csv("drive/MyDrive/cell_gnn_probs.csv", index=False)

print("Saved: drive/MyDrive/cell_gnn_probs.csv")
